# 说明
> 1. 部分输出太长且没啥必要所以我做了清除输出操作
> 2. 我的主要任务是在paddlepaddle框架下复刻video swin transformer，并在搜集的暴力视频数据下训练识别暴力行为，为了提高开发效率(尤其是想借助其底层的多卡训练、混合精度加速等)，于是在paddlevideo组件库下进一步开发的，导致我很多并不是在notebook block内敲的，采取了附上源代码和配置文件的形式，望理解。
>> 1.多卡训练由于notebook不支持多线程，所以train是通过终端调用py实现。
>>
>> 2.PaddleVideo利用依赖注入技术实现控制反转，来对整个系统进行解耦，通过可自定义调整的配置文件来控制整个系统从而实现模块化。                                                                                  

# 搭建环境

## aistudio平台相关
>> 由于这个swin transformer计算量属实太大了，而且刚好是用的paddlepaddle的框架，于是用的是百度的aistudio（价格还算是实惠）

请点击[此处](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576)查看本环境基本用法.  <br>
Please click [here ](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576) for more detailed instructions. 

In [1]:
# 同时添加如下代码, 这样每次环境(kernel)启动的时候只要运行下方代码即可: 
import sys 
sys.path.append('/home/aistudio/external-libraries')

In [1]:
# 如果需要进行持久化安装, 需要使用持久化路径, 如下方代码示例:
!mkdir /home/aistudio/external-libraries
!pip install beautifulsoup4 -t /home/aistudio/external-libraries

mkdir: cannot create directory '/home/aistudio/external-libraries': File exists
ERROR: Can not combine '--user' and '--target'


## git库配置
>> 我这个题目出自我的大创项目，当时我们是用的GitHub，但是在这个codelab中因为网络问题，遂改用的gitee

In [4]:
! git clone https://gitee.com/IronHxs/huitong-zhidun.git

fatal: destination path 'huitong-zhidun' already exists and is not an empty directory.


In [ ]:
! git config --global user.email "3077066784@qq.com"
! git config --global user.name "Iron.hxs"

In [3]:
! apt install unzip git

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


## 库相关
>> 我是在paddle video这个组件库下进行的开发和训练，由于我对这个库进行了改动，所以我是直接将他download下来，进行了改动|

In [10]:
! python -m pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/, https://mirrors.aliyun.com/pypi/simple/


In [2]:
! unzip -d /home/aistudio/data /home/aistudio/huitong-zhidun/Swin_video/PaddleVideo-develop.zip

  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/data/download_features.sh  
   creating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/data_loader/
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/data_loader/MSRVTT_dataset.py  
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/data_loader/data_loaders.py  
   creating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/imgs/
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/imgs/t2vlad.png  
   creating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/logger/
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/logger/__init__.py  
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/logger/log_parser.py  
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD/logger/logger.py  
  inflating: /home/aistudio/data/PaddleVideo-develop/applications/T2VLAD

> paddlevideo框架中用的numpy是老版本，其中使用了np.int,为避免每次解压paddlevideo手动更改的麻烦，遂写了sh文件，每次解压完后进行相应一些修改。

In [7]:
! chmod +x /home/aistudio/huitong-zhidun/Swin_video/fix_paddlevideo.sh
! /home/aistudio/huitong-zhidun/Swin_video/fix_paddlevideo.sh

代码已替换为动态适配版本
完成 np.int -> int 替换
原始文件从top1和5改为仅top1


>相关的一些依赖项安装

In [4]:
! python install_deps.py

安装 rarfile...
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
rarfile 安装成功!
安装 decord...
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
decord 安装成功!
安装 paddlepaddle>=2.3.1...
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
paddlepaddle>=2.3.1 安装成功!
所有依赖项安装成功!


>设备检查

In [1]:
! python -c "import paddle; print(paddle.device.is_compiled_with_cuda()); print(paddle.device.cuda.device_count())"

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
True
4


## 整合数据集

> 搜集了多个相关暴力数据集，此处进行了整合生成了train、test、val的三个list文件，方便在paddlevideo的框架下快速开始训练。

In [2]:
! python  /home/aistudio/huitong-zhidun/Swin_video/extract_datasets.py

安装必要的依赖...
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
解压 /home/aistudio/data/data341189/-violent1.rar 到 /home/aistudio/data/violence_dataset/violent1
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
已成功使用rarfile库解压 /home/aistudio/data/data341189/-violent1.rar
解压 /home/aistudio/data/data341189/-violent2.rar 到 /home/aistudio/data/violence_dataset/violent2
Looking in indexes: https://mirror.baidu.com/pypi/simple/, https://mirrors.aliyun.com/pypi/simple/
已成功使用rarfile库解压 /home/aistudio/data/data341189/-violent2.rar
解压 /home/aistudio/data/data341189/HockeyFightVidoes.zip 到 /home/aistudio/data/violence_dataset/HockeyFightVidoes
已成功使用zipfile模块解压 /home/aistudio/data/data341189/HockeyFightVidoes.zip
解压 /home/aistudio/data/data341189/RealLifeViolenceSituationsDataset.zip 到 /home/aistudio/data/violence_dataset/RealLifeViolenceSituationsDataset
已成功使用zipfile模块解压 /home/aistudio/data/data341189/Re

> 提取数据集的py代码（extract_datasets.py）附上如下：

In [ ]:
import os
import zipfile
import subprocess
import shutil
from pathlib import Path
import random
import time

def extract_zip(zip_path, extract_to, attempt=0):
    """使用多种方法尝试解压ZIP文件"""
    print(f"解压 {zip_path} 到 {extract_to}")
    os.makedirs(extract_to, exist_ok=True)
    
    # 方法1: Python zipfile模块
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"已成功使用zipfile模块解压 {zip_path}")
        return True
    except NotImplementedError:
        print(f"ZIP文件 {zip_path} 使用了不支持的压缩方法，尝试其他方式解压...")
    except Exception as e:
        print(f"使用zipfile解压时出错: {str(e)}")
    
    # 方法2: 使用unzip命令行工具(如果可用)
    try:
        result = subprocess.run(['unzip', '-o', '-q', zip_path, '-d', extract_to], 
                               stderr=subprocess.PIPE, check=False)
        if result.returncode == 0:
            print(f"已成功使用unzip命令解压 {zip_path}")
            return True
        else:
            print(f"unzip命令失败: {result.stderr.decode()}")
    except Exception as e:
        print(f"尝试使用unzip命令时出错: {str(e)}")
    
    # 方法3: 使用Python subprocess调用系统解压工具
    try:
        if os.name == 'nt':  # Windows
            # 在Windows上尝试使用7z
            result = subprocess.run(['7z', 'x', '-y', f'-o{extract_to}', zip_path], 
                                   stderr=subprocess.PIPE, check=False)
        else:  # Linux/Mac
            # 在AIStudio环境中，尝试使用Python执行解压
            # 确保我们已安装解压工具
            if attempt == 0:  # 防止无限递归
                try:
                    subprocess.run(['pip', 'install', '--user', 'pyunpack', 'patool'], check=True)
                    time.sleep(2)  # 等待安装完成
                    return extract_zip(zip_path, extract_to, attempt=1)
                except Exception as e:
                    print(f"安装pyunpack失败: {str(e)}")
            
            try:
                from pyunpack import Archive
                Archive(zip_path).extractall(extract_to)
                print(f"已成功使用pyunpack解压 {zip_path}")
                return True
            except Exception as e:
                print(f"使用pyunpack解压失败: {str(e)}")
                
                # 最后尝试使用jar命令
                result = subprocess.run(['jar', 'xf', zip_path], 
                                        cwd=extract_to,
                                        stderr=subprocess.PIPE, check=False)
                
        if 'result' in locals() and result.returncode == 0:
            print(f"已成功使用系统命令解压 {zip_path}")
            return True
        elif 'result' in locals():
            print(f"系统命令解压失败: {result.stderr.decode() if hasattr(result, 'stderr') else 'Unknown error'}")
    except Exception as e:
        print(f"尝试使用系统命令时出错: {str(e)}")
    
    print(f"警告: 无法解压 {zip_path}，所有方法都失败了")
    return False

def extract_rar(rar_path, extract_to):
    """尝试使用多种方式解压RAR文件"""
    print(f"解压 {rar_path} 到 {extract_to}")
    os.makedirs(extract_to, exist_ok=True)
    
    # 方法1: 尝试使用unrar命令
    try:
        result = subprocess.run(['unrar', 'x', '-o+', rar_path, extract_to], 
                               stderr=subprocess.PIPE, check=False)
        if result.returncode == 0:
            print(f"已成功使用unrar命令解压 {rar_path}")
            return True
    except Exception:
        pass
    
    # 方法2: 尝试安装和使用rarfile库
    try:
        subprocess.check_call(['pip', 'install', '--user', 'rarfile'])
        import rarfile
        with rarfile.RarFile(rar_path) as rf:
            rf.extractall(extract_to)
        print(f"已成功使用rarfile库解压 {rar_path}")
        return True
    except Exception as e:
        print(f"使用rarfile解压失败: {str(e)}")
    
    # 方法3: 尝试使用7z(如果可用)
    try:
        if os.name == 'nt':  # Windows
            cmd = ['7z', 'x', '-y', f'-o{extract_to}', rar_path]
        else:  # Linux/Mac
            cmd = ['7z', 'x', '-y', f'-o{extract_to}', rar_path]
        
        result = subprocess.run(cmd, stderr=subprocess.PIPE, check=False)
        if result.returncode == 0:
            print(f"已成功使用7z解压 {rar_path}")
            return True
        else:
            print(f"7z解压失败: {result.stderr.decode() if hasattr(result, 'stderr') else 'Unknown error'}")
    except Exception as e:
        print(f"尝试使用7z时出错: {str(e)}")
    
    # 方法4: 尝试使用Python的pyunpack库
    try:
        subprocess.run(['pip', 'install', '--user', 'pyunpack', 'patool'], check=False)
        from pyunpack import Archive
        Archive(rar_path).extractall(extract_to)
        print(f"已成功使用pyunpack解压 {rar_path}")
        return True
    except Exception as e:
        print(f"使用pyunpack解压失败: {str(e)}")
    
    print(f"警告: 无法解压 {rar_path}，所有方法都失败了")
    return False

def process_datasets(data_dir, output_dir):
    """处理所有数据集文件"""
    os.makedirs(output_dir, exist_ok=True)
    
    # 定义所有数据集文件
    datasets = {
        "-violent1.rar": "violent1",
        "-violent2.rar": "violent2",
        "HockeyFightVidoes.zip": "HockeyFightVidoes",
        "RealLifeViolenceSituationsDataset.zip": "RealLifeViolenceSituationsDataset",
        "SurveillanceCameraFightDataset.zip": "SurveillanceCameraFightDataset",
        "fight-detection-surv-dataset-master.zip": "fight-detection-surv-dataset-master",
        "non-violent.rar": "non-violent",
        "annotation.zip": "annotation",
        # "fight.zip": "fight",
        "normal_1.zip": "normal_1",
        "normal_2.zip": "normal_2"
    }
    
    success_count = 0
    for filename, target_dir in datasets.items():
        file_path = os.path.join(data_dir, filename)
        extract_path = os.path.join(output_dir, target_dir)
        
        if not os.path.exists(file_path):
            print(f"警告: 文件不存在 {file_path}")
            continue
        
        if filename.lower().endswith('.zip'):
            if extract_zip(file_path, extract_path):
                success_count += 1
        elif filename.lower().endswith('.rar'):
            if extract_rar(file_path, extract_path):
                success_count += 1
        else:
            print(f"不支持的文件格式: {filename}")
    
    print(f"成功处理了 {success_count}/{len(datasets)} 个数据集文件")
    return success_count > 0

def find_video_files(base_dir):
    """查找所有视频文件"""
    video_files = []
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv']
    
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in video_extensions):
                full_path = os.path.abspath(os.path.join(root, file))
                video_files.append(full_path)
    
    return video_files

def create_dataset_lists(base_dir, output_dir):
    """创建训练、验证和测试列表"""
    # 收集所有视频文件
    all_videos = find_video_files(base_dir)
    print(f"找到总共 {len(all_videos)} 个视频文件")
    
    if len(all_videos) == 0:
        print("错误: 未找到视频文件!")
        return 0, 0, 0
    
    # 确定视频标签
    violence_videos = []
    non_violence_videos = []
    

    
    for video_path in all_videos:
        from pathlib import Path
        p = Path(video_path)
        parts = [part.lower() for part in p.parts]
        file_name = p.name
        # 若路径中包含 violent1 或 violent2 目录，或文件名以 F, f, V 开头，则视为暴力
        if any(dir_name in ("violent1", "violent2") for dir_name in parts) or file_name.startswith(('F', 'f', 'V')):
            violence_videos.append(f"{video_path} 1")
        else:
            non_violence_videos.append(f"{video_path} 0")
    print(f"分类结果: {len(violence_videos)} 个暴力视频, {len(non_violence_videos)} 个非暴力视频")
    
    # 打乱并分割数据
    random.seed(42)  # 设置随机种子确保可重复性
    random.shuffle(violence_videos)
    random.shuffle(non_violence_videos)
    
    # 80% 训练, 10% 验证, 10% 测试
    train_ratio, val_ratio = 0.8, 0.1
    
    train_violence = violence_videos[:int(len(violence_videos) * train_ratio)]
    val_violence = violence_videos[int(len(violence_videos) * train_ratio):
                                   int(len(violence_videos) * (train_ratio + val_ratio))]
    test_violence = violence_videos[int(len(violence_videos) * (train_ratio + val_ratio)):]
    
    train_non_violence = non_violence_videos[:int(len(non_violence_videos) * train_ratio)]
    val_non_violence = non_violence_videos[int(len(non_violence_videos) * train_ratio):
                                          int(len(non_violence_videos) * (train_ratio + val_ratio))]
    test_non_violence = non_violence_videos[int(len(non_violence_videos) * (train_ratio + val_ratio)):]
    
    # 合并并保存
    train_list = train_violence + train_non_violence
    val_list = val_violence + val_non_violence
    test_list = test_violence + test_non_violence
    
    random.shuffle(train_list)
    random.shuffle(val_list)
    random.shuffle(test_list)
    
    os.makedirs(output_dir, exist_ok=True)
    
    with open(os.path.join(output_dir, 'train_list.txt'), 'w') as f:
        f.write('\n'.join(train_list))
    
    with open(os.path.join(output_dir, 'val_list.txt'), 'w') as f:
        f.write('\n'.join(val_list))
    
    with open(os.path.join(output_dir, 'test_list.txt'), 'w') as f:
        f.write('\n'.join(test_list))
    
    print(f"创建训练集列表：{len(train_list)}条")
    print(f"创建验证集列表：{len(val_list)}条")
    print(f"创建测试集列表：{len(test_list)}条")
    
    # 创建类别映射文件
    with open(os.path.join(output_dir, 'label_map.txt'), 'w') as f:
        f.write('0 non-violence\n1 violence')
    
    return len(train_list), len(val_list), len(test_list)



def main():
    # 基础目录
    data_dir = "/home/aistudio/data/data341189"  # 压缩文件所在目录
    extract_dir = "/home/aistudio/data/violence_dataset"  # 解压目标目录
    lists_dir = "/home/aistudio/data/violence_dataset/list"  # 列表文件保存目录
    
    # 安装必要的依赖
    print("安装必要的依赖...")
    subprocess.run(["pip", "install", "--user", "decord"], check=False)
    
    # 解压数据集
    if process_datasets(data_dir, extract_dir):
        print("数据集解压完成")
    else:
        print("警告: 数据集解压过程中存在问题")
    
    # 创建数据集列表
    train_count, val_count, test_count = create_dataset_lists(extract_dir, lists_dir)
    
    # 创建配置文件
    config_path = create_config_file(config_dir)
    
    print("\n数据准备完成!")
    print(f"训练集: {train_count} 个视频")
    print(f"验证集: {val_count} 个视频")
    print(f"测试集: {test_count} 个视频")
if __name__ == "__main__":
    main()

>本打算预提取出帧，然后方便提高训练效率，但是在实际extract时发现存储直接爆炸（大于1个T），遂放弃。

In [8]:
! python extract_rawframes.py /home/aistudio/data/violence_dataset/ ./frames/ --ext mp4 --num_worker 16

2025-06-02 20:09:17,806 - INFO - 创建输出目录: ./frames/
2025-06-02 20:09:17,806 - INFO - 创建类别目录: ./frames/violent2
2025-06-02 20:09:17,806 - INFO - 创建类别目录: ./frames/fight-detection-surv-dataset-master
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/RealLifeViolenceSituationsDataset
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/non-violent
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/.ipynb_checkpoints
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/normal_2
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/violent1
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/annotation
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/SurveillanceCameraFightDataset
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/list
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/HockeyFightVidoes
2025-06-02 20:09:17,807 - INFO - 创建类别目录: ./frames/normal_1
2025-06-02 20:09:17,807 - INFO - 从 /home/aistudio/data/violence_dataset/ 中查找视频文件 (扩展名: mp4)
2025-06-02 20:09:17,810 - INFO - 找到 784 个视频文件
2025-06-0

## 模型开发

>附我的swin transformer 的模型代码

In [ ]:
from functools import lru_cache, reduce
from operator import mul

import numpy as np
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.nn.initializer import Constant

from ...utils import load_ckpt
from ..registry import BACKBONES
from ..weight_init import trunc_normal_

zeros_ = Constant(value=0.)
ones_ = Constant(value=1.)


def drop_path(x, drop_prob=0., training=False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = paddle.to_tensor(1 - drop_prob)
    shape = (x.shape[0], ) + (1, ) * (x.ndim - 1)
    random_tensor = keep_prob + paddle.rand(shape, dtype=x.dtype)
    random_tensor = paddle.floor(random_tensor)  # binarize
    output = x.divide(keep_prob) * random_tensor

    return output


class DropPath(nn.Layer):
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class Mlp(nn.Layer):
    def __init__(self,
                 in_features,
                 hidden_features=None,
                 out_features=None,
                 act_layer=nn.GELU,
                 drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def window_partition(x, window_size):
    B, D, H, W, C = x.shape
    x = x.reshape([
        B, D // window_size[0], window_size[0], H // window_size[1],
        window_size[1], W // window_size[2], window_size[2], C
    ])
    windows = x.transpose([0, 1, 3, 5, 2, 4, 6,
                           7]).reshape([-1, reduce(mul, window_size), C])
    return windows


class Identity(nn.Layer):
    def __init__(self):
        super(Identity, self).__init__()

    def forward(self, input):
        return input


def window_reverse(windows, window_size, B, D, H, W):
    x = windows.reshape([
        B, D // window_size[0], H // window_size[1], W // window_size[2],
        window_size[0], window_size[1], window_size[2], -1
    ])
    x = x.transpose([0, 1, 4, 2, 5, 3, 6, 7]).reshape([B, D, H, W, -1])
    return x


def get_window_size(x_size, window_size, shift_size=None):
    use_window_size = list(window_size)
    if shift_size is not None:
        use_shift_size = list(shift_size)
    for i in range(len(x_size)):
        if x_size[i] <= window_size[i]:
            use_window_size[i] = x_size[i]
            if shift_size is not None:
                use_shift_size[i] = 0

    if shift_size is None:
        return tuple(use_window_size)
    else:
        return tuple(use_window_size), tuple(use_shift_size)


class WindowAttention3D(nn.Layer):
    def __init__(self,
                 dim,
                 window_size,
                 num_heads,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop=0.,
                 proj_drop=0.):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wd, Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim**-0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = self.create_parameter(
            shape=((2 * window_size[0] - 1) * (2 * window_size[1] - 1) *
                   (2 * window_size[2] - 1), num_heads),
            default_initializer=zeros_,
        )  # 2*Wd-1 * 2*Wh-1 * 2*Ww-1, nH
        self.add_parameter("relative_position_bias_table",
                           self.relative_position_bias_table)
        # get pair-wise relative position index for each token inside the window
        coords_d = paddle.arange(self.window_size[0])
        coords_h = paddle.arange(self.window_size[1])
        coords_w = paddle.arange(self.window_size[2])
        coords = paddle.stack(paddle.meshgrid(coords_d, coords_h,
                                              coords_w))  # 3, Wd, Wh, Ww
        coords_flatten = paddle.flatten(coords, 1)  # 3, Wd*Wh*Ww

        relative_coords = coords_flatten.unsqueeze(
            axis=2) - coords_flatten.unsqueeze(axis=1)  # 3, Wd*Wh*Ww, Wd*Wh*Ww

        # relative_coords = coords_flatten.unsqueeze(2) - coords_flatten.unsqueeze(1)  # 3, Wd*Wh*Ww, Wd*Wh*Ww
        relative_coords = relative_coords.transpose([1, 2, 0
                                                     ])  # Wd*Wh*Ww, Wd*Wh*Ww, 3
        relative_coords[:, :,
                        0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 2] += self.window_size[2] - 1

        relative_coords[:, :, 0] *= (2 * self.window_size[1] -
                                     1) * (2 * self.window_size[2] - 1)
        relative_coords[:, :, 1] *= (2 * self.window_size[2] - 1)
        relative_position_index = relative_coords.sum(
            axis=-1)  # Wd*Wh*Ww, Wd*Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias_attr=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=0.02)
        self.softmax = nn.Softmax(axis=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(
            [B_, N, 3, self.num_heads,
             C // self.num_heads]).transpose([2, 0, 3, 1, 4])
        q, k, v = qkv[0], qkv[1], qkv[2]  # B_, nH, N, C

        q = q * self.scale
        attn = q @ k.transpose([0, 1, 3, 2])

        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index[:N, :N].reshape([-1])].reshape(
                [N, N, -1])  # Wd*Wh*Ww,Wd*Wh*Ww,nH
        relative_position_bias = relative_position_bias.transpose(
            [2, 0, 1])  # nH, Wd*Wh*Ww, Wd*Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)  # B_, nH, N, N

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.reshape([B_ // nW, nW, self.num_heads, N, N
                                 ]) + mask.unsqueeze(1).unsqueeze(0).astype(attn.dtype)
            attn = attn.reshape([-1, self.num_heads, N, N])
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose([0, 2, 1, 3]).reshape([B_, N, C])
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock3D(nn.Layer)
    def __init__(self,
                 dim,
                 num_heads,
                 window_size=(2, 7, 7),
                 shift_size=(0, 0, 0),
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm,
                 use_checkpoint=False):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        # self.use_checkpoint=use_checkpoint

        assert 0 <= self.shift_size[0] < self.window_size[
            0], "shift_size must in 0-window_size"
        assert 0 <= self.shift_size[1] < self.window_size[
            1], "shift_size must in 0-window_size"
        assert 0 <= self.shift_size[2] < self.window_size[
            2], "shift_size must in 0-window_size"

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention3D(dim,
                                      window_size=self.window_size,
                                      num_heads=num_heads,
                                      qkv_bias=qkv_bias,
                                      qk_scale=qk_scale,
                                      attn_drop=attn_drop,
                                      proj_drop=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim,
                       hidden_features=mlp_hidden_dim,
                       act_layer=act_layer,
                       drop=drop)

    def forward_part1(self, x, mask_matrix):
        B = x.shape[0]
        _, D, H, W, C = x.shape
        window_size, shift_size = get_window_size((D, H, W), self.window_size,
                                                  self.shift_size)

        x = self.norm1(x)
        # pad feature maps to multiples of window size
        pad_l = pad_t = pad_d0 = 0
        pad_d1 = (window_size[0] - D % window_size[0]) % window_size[0]
        pad_b = (window_size[1] - H % window_size[1]) % window_size[1]
        pad_r = (window_size[2] - W % window_size[2]) % window_size[2]
        x = F.pad(x, (pad_l, pad_r, pad_t, pad_b, pad_d0, pad_d1),
                  data_format='NDHWC')
        _, Dp, Hp, Wp, _ = x.shape
        # cyclic shift
        if any(i > 0 for i in shift_size):
            shifted_x = paddle.roll(x,
                                    shifts=(-shift_size[0], -shift_size[1],
                                            -shift_size[2]),
                                    axis=(1, 2, 3))
            attn_mask = mask_matrix
        else:
            shifted_x = x
            attn_mask = None
        # partition windows
        x_windows = window_partition(shifted_x,
                                     window_size)  # B*nW, Wd*Wh*Ww, C
        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=attn_mask)  # B*nW, Wd*Wh*Ww, C
        # merge windows
        attn_windows = attn_windows.reshape([-1, *(window_size + (C, ))])
        shifted_x = window_reverse(attn_windows, window_size, B, Dp, Hp,
                                   Wp)  # B D' H' W' C
        # reverse cyclic shift
        if any(i > 0 for i in shift_size):
            x = paddle.roll(shifted_x,
                            shifts=(shift_size[0], shift_size[1],
                                    shift_size[2]),
                            axis=(1, 2, 3))
        else:
            x = shifted_x

        if pad_d1 > 0 or pad_r > 0 or pad_b > 0:
            x = x[:, :D, :H, :W, :]
        return x

    def forward_part2(self, x):
        return self.drop_path(self.mlp(self.norm2(x)))

    def forward(self, x, mask_matrix):
        shortcut = x
        x = self.forward_part1(x, mask_matrix)
        x = shortcut + self.drop_path(x).astype(shortcut.dtype)
        x = x + self.forward_part2(x).astype(x.dtype)

        return x


class PatchMerging(nn.Layer):
    def __init__(self, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias_attr=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x):
        B, D, H, W, C = x.shape

        # padding
        pad_input = (H % 2 == 1) or (W % 2 == 1)
        if pad_input:
            x = F.pad(x, (0, W % 2, 0, H % 2, 0, 0), data_format='NDHWC')

        x0 = x[:, :, 0::2, 0::2, :]  # B D H/2 W/2 C
        x1 = x[:, :, 1::2, 0::2, :]  # B D H/2 W/2 C
        x2 = x[:, :, 0::2, 1::2, :]  # B D H/2 W/2 C
        x3 = x[:, :, 1::2, 1::2, :]  # B D H/2 W/2 C
        x = paddle.concat([x0, x1, x2, x3], -1)  # B D H/2 W/2 4*C

        x = self.norm(x)
        x = self.reduction(x)

        return x


# cache each stage results
@lru_cache()
def compute_mask(D, H, W, window_size, shift_size):
    img_mask = paddle.zeros((1, D, H, W, 1))  # 1 Dp Hp Wp 1
    cnt = 0
    for d in slice(-window_size[0]), slice(-window_size[0],
                                           -shift_size[0]), slice(
                                               -shift_size[0], None):
        for h in slice(-window_size[1]), slice(-window_size[1],
                                               -shift_size[1]), slice(
                                                   -shift_size[1], None):
            for w in slice(-window_size[2]), slice(-window_size[2],
                                                   -shift_size[2]), slice(
                                                       -shift_size[2], None):
                img_mask[:, d, h, w, :] = cnt
                cnt += 1
    mask_windows = window_partition(img_mask,
                                    window_size)  # nW, ws[0]*ws[1]*ws[2], 1
    mask_windows = mask_windows.squeeze(-1)  # nW, ws[0]*ws[1]*ws[2]
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    # attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
    huns = -100.0 * paddle.ones_like(attn_mask)
    attn_mask = huns * (attn_mask != 0).astype("float32")
    return attn_mask


class BasicLayer(nn.Layer):
    def __init__(self,
                 dim,
                 depth,
                 num_heads,
                 window_size=(1, 7, 7),
                 mlp_ratio=4.,
                 qkv_bias=False,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 norm_layer=nn.LayerNorm,
                 downsample=None,
                 use_checkpoint=False):
        super().__init__()
        self.window_size = window_size
        self.shift_size = tuple(i // 2 for i in window_size)
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.LayerList([
            SwinTransformerBlock3D(
                dim=dim,
                num_heads=num_heads,
                window_size=window_size,
                shift_size=(0, 0, 0) if (i % 2 == 0) else self.shift_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop=drop,
                attn_drop=attn_drop,
                drop_path=drop_path[i]
                if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer,
                use_checkpoint=use_checkpoint,
            ) for i in range(depth)
        ])

        self.downsample = downsample
        if self.downsample is not None:
            self.downsample = downsample(dim=dim, norm_layer=norm_layer)

    def forward(self, x):
        # calculate attention mask for SW-MSA
        B = x.shape[0]
        _, C, D, H, W = x.shape
        window_size, shift_size = get_window_size((D, H, W), self.window_size,
                                                  self.shift_size)
        # x = rearrange(x, 'b c d h w -> b d h w c')
        x = x.transpose([0, 2, 3, 4, 1])
        Dp = int(np.ceil(D / window_size[0])) * window_size[0]
        Hp = int(np.ceil(H / window_size[1])) * window_size[1]
        Wp = int(np.ceil(W / window_size[2])) * window_size[2]
        attn_mask = compute_mask(Dp, Hp, Wp, window_size, shift_size)
        for blk in self.blocks:
            x = blk(x, attn_mask)
        x = x.reshape([B, D, H, W, C])

        if self.downsample is not None:
            x = self.downsample(x)
        x = x.transpose([0, 4, 1, 2, 3])
        return x


class PatchEmbed3D(nn.Layer):
    def __init__(self,
                 patch_size=(2, 4, 4),
                 in_chans=3,
                 embed_dim=96,
                 norm_layer=None):
        super().__init__()
        self.patch_size = patch_size

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.proj = nn.Conv3D(in_chans,
                              embed_dim,
                              kernel_size=patch_size,
                              stride=patch_size)
        if norm_layer is not None:
            self.norm = norm_layer(embed_dim)
        else:
            self.norm = None

    def forward(self, x):
        _, _, D, H, W = x.shape
        if W % self.patch_size[2] != 0:
            x = F.pad(
                x, (0, self.patch_size[2] - W % self.patch_size[2], 0, 0, 0, 0),
                data_format='NCDHW')
        if H % self.patch_size[1] != 0:
            x = F.pad(
                x, (0, 0, 0, self.patch_size[1] - H % self.patch_size[1], 0, 0),
                data_format='NCDHW')
        if D % self.patch_size[0] != 0:
            x = F.pad(
                x, (0, 0, 0, 0, 0, self.patch_size[0] - D % self.patch_size[0]),
                data_format='NCDHW')

        x = self.proj(x)  # B C D Wh Ww
        if self.norm is not None:
            D, Wh, Ww = x.shape[2], x.shape[3], x.shape[4]
            x = x.flatten(2).transpose([0, 2, 1])
            x = self.norm(x)
            x = x.transpose([0, 2, 1]).reshape([-1, self.embed_dim, D, Wh, Ww])

        return x


@BACKBONES.register()
class SwinTransformer3D(nn.Layer):
    def __init__(self,
                 pretrained=None,
                 patch_size=(4, 4, 4),
                 in_chans=3,
                 embed_dim=96,
                 depths=[2, 2, 6, 2],
                 num_heads=[3, 6, 12, 24],
                 window_size=(2, 7, 7),
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop_rate=0.,
                 attn_drop_rate=0.,
                 drop_path_rate=0.2,
                 norm_layer=nn.LayerNorm,
                 patch_norm=False,
                 frozen_stages=-1,
                 use_checkpoint=False):
        super().__init__()

        self.pretrained = pretrained
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.patch_norm = patch_norm
        self.frozen_stages = frozen_stages
        self.window_size = window_size
        self.patch_size = patch_size

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed3D(
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [
            x.item() for x in paddle.linspace(0, drop_path_rate, sum(depths))
        ]  # stochastic depth decay rule

        # build layers
        self.layers = nn.LayerList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(
                dim=int(embed_dim * 2**i_layer),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer,
                downsample=PatchMerging
                if i_layer < self.num_layers - 1 else None,
                use_checkpoint=use_checkpoint)
            self.layers.append(layer)

        self.num_features = int(embed_dim * 2**(self.num_layers - 1))

        # add a norm layer for each output
        self.norm = norm_layer(self.num_features)

        self._freeze_stages()

    def _freeze_stages(self):
        if self.frozen_stages >= 0:
            self.patch_embed.eval()
            for param in self.patch_embed.parameters():
                param.stop_gradient = True

        if self.frozen_stages >= 1:
            self.pos_drop.eval()
            for i in range(0, self.frozen_stages):
                m = self.layers[i]
                m.eval()
                for param in m.parameters():
                    param.stop_gradient = True

    def _init_fn(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            zeros_(m.bias)
            ones_(m.weight)

    def init_weights(self):

        self.apply(self._init_fn)
        if isinstance(
                self.pretrained, str
        ) and self.pretrained.strip() != "":  # load pretrained weights
            load_ckpt(self, self.pretrained)
        elif self.pretrained is None or self.pretrained.strip() == "":
            pass
        else:
            raise NotImplementedError

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x)

        x = x.transpose([0, 2, 3, 4, 1])
        x = self.norm(x)
        x = x.transpose([0, 4, 1, 2, 3])
        return x

    def train(self, mode=True):
        super(SwinTransformer3D, self).train(mode)
        self._freeze_stages()


## 训练
>该组件库利用依赖注入技术实现控制反转，来对整个系统进行解耦，通过可自定义调整的配置文件来控制整个系统从而实现模块化，所以显式训练的代码没有，主要是通过配置yaml文件实现，附上我的yaml文件如下。

MODEL: #MODEL field
    framework: "RecognizerTransformer" #Mandatory, indicate the type of network, associate to the 'paddlevideo/modeling/framework/' .
    backbone: #Mandatory, indicate the type of backbone, associate to the 'paddlevideo/modeling/backbones/' .
        name: "SwinTransformer3D" #Mandatory, The name of backbone.
        pretrained: "/home/aistudio/data/swin_base_patch4_window7_224.pdparams" #Optional, pretrained model path.
        patch_size: [2, 4, 4]
        embed_dim: 128
        depths: [2, 2, 18, 2]
        num_heads: [4, 8, 16, 32]
        window_size: [8, 7, 7]
        mlp_ratio: 4.
        qkv_bias: True
        qk_scale: None
        drop_rate: 0.0
        attn_drop_rate: 0.0
        drop_path_rate: 0.2
        patch_norm: True
    head:
        name: "I3DHead" #Mandatory, indicate the type of head, associate to the 'paddlevideo/modeling/heads'
        num_classes: 400 #Optional, the number of classes to be classified.
        in_channels: 1024 #input channel of the extracted feature.
        spatial_type: "avg"
        drop_ratio: 0.5 #the ratio of dropout
        std: 0.01 #std value in params initialization
    runtime_cfg: # configuration used when the model is train or test.
        test: # test config
            num_seg: 32
            avg_type: "prob" # 'score' or 'prob'

DATASET: #DATASET field
    batch_size: 1 #Mandatory, bacth size
    num_workers: 36 #Mandatory, XXX the number of subprocess on each GPU.
    test_batch_size: 1
    train:
        format: "VideoDataset" #Mandatory, indicate the type of dataset, associate to the 'paddlevidel/loader/dateset'
        data_prefix: "" #Mandatory, train data root path
        file_path: "/home/aistudio/data/violence_dataset/list/train_list.txt" #Mandatory, train data index file path
    valid:
        format: "VideoDataset" #Mandatory, indicate the type of dataset, associate to the 'paddlevidel/loader/dateset'
        data_prefix: "" #Mandatory, train data root path
        file_path: "/home/aistudio/data/violence_dataset/list/val_list.txt" #Mandatory, valid data index file path
    test:
        format: "VideoDataset" #Mandatory, indicate the type of dataset, associate to the 'paddlevidel/loader/dateset'
        data_prefix: "" #Mandatory, train data root path
        file_path: "/home/aistudio/data/violence_dataset/list/test_list.txt" #Mandatory, valid data index file path

PIPELINE: #PIPELINE field TODO.....
    train: #Mandotary, indicate the pipeline to deal with the training data, associate to the 'paddlevideo/loader/pipelines/'
        decode:
            name: "VideoDecoder"
            backend: "decord"
            mode: "train"
        sample:
            name: "Sampler"
            num_seg: 1
            frame_interval: 2
            seg_len: 32
            valid_mode: False
            use_pil: False
        transform: #Mandotary, image transform operator.
            - Scale:
                  short_size: 256
                  fixed_ratio: False
                  keep_ratio: True
                  backend: "cv2"
                  do_round: True
            - RandomResizedCrop:
                  backend: "cv2"
            - Scale:
                  short_size: 224
                  fixed_ratio: False
                  keep_ratio: False
                  backend: "cv2"
                  do_round: True
            - RandomFlip:
            - Normalization:
                  mean: [123.675, 116.28, 103.53]
                  std: [58.395, 57.12, 57.375]
                  tensor_shape: [3, 1, 1, 1]
                  inplace: True
            - Image2Array:
                  data_format: "cthw"
    valid: #Mandatory, indicate the pipeline to deal with the validing data. associate to the 'paddlevideo/loader/pipelines/'
        decode:
            name: "VideoDecoder"
            backend: "decord"
            mode: "valid"
        sample:
            name: "Sampler"
            num_seg: 1
            frame_interval: 2
            seg_len: 32
            valid_mode: True
            use_pil: False
        transform: #Mandotary, image transform operator.
            - Scale:
                  short_size: 256
                  fixed_ratio: False
                  keep_ratio: True
                  backend: "cv2"
                  do_round: True
            - CenterCrop:
                  target_size: 224
                  do_round: False
                  backend: "cv2"
            - Normalization:
                  mean: [123.675, 116.28, 103.53]
                  std: [58.395, 57.12, 57.375]
                  tensor_shape: [3, 1, 1, 1]
                  inplace: True
            - Image2Array:
                  data_format: "cthw"
    test:
        decode:
            name: "VideoDecoder"
            backend: "decord"
            mode: "valid"
        sample:
            name: "Sampler"
            num_seg: 4
            frame_interval: 2
            seg_len: 32
            valid_mode: True
            use_pil: False
        transform: #Mandotary, image transform operator.
            - Scale:
                  short_size: 224
                  fixed_ratio: False
                  keep_ratio: True
                  backend: "cv2"
                  do_round: True
            - UniformCrop:
                  target_size: 224
                  backend: "cv2"
            - Normalization:
                  mean: [123.675, 116.28, 103.53]
                  std: [58.395, 57.12, 57.375]
                  tensor_shape: [3, 1, 1, 1]
                  inplace: True
            - Image2Array:
                  data_format: "cthw"

OPTIMIZER: #OPTIMIZER field
    name: "AdamW" #Mandatory, the type of optimizer, associate to the 'paddlevideo/solver/'
    beta1: 0.9
    beta2: 0.999
    no_weight_decay_name: "norm relative_position_bias_table"
    learning_rate: #Mandatory, the type of learning rate scheduler, associate to the 'paddlevideo/solver/'
        name: "CustomWarmupCosineStepDecay"
        iter_step: True
        warmup_iters: 2.5
        warmup_ratio: 0.1
        min_lr: 0
        base_lr: 3e-5
        max_epoch: 30
    weight_decay: 0.05

METRIC:
    name: "CenterCropMetric"

GRADIENT_ACCUMULATION:
    global_batch_size: 64 # Specify the sum of batches to be calculated by all GPUs

INFERENCE:
    name: "VideoSwin_Inference_helper"
    num_seg: 1
    seg_len: 32
    short_size: 256
    target_size: 224

model_name: "VideoSwin_base"
log_interval: 20 #Optional, the interal of logger, default:10
save_interval: 5
epochs: 30 #Mandatory, total epoch
log_level: "INFO" #Optional, the logger level. default: "INFO"

In [8]:
! export CUDA_VISIBLE_DEVICES=0,1,2,3
! export FLAGS_conv_workspace_size_limit=800 # MB
! export FLAGS_cudnn_exhaustive_search=1
! export FLAGS_cudnn_batchnorm_spatial_persistent=1
! python3.10 -u -B -m paddle.distributed.launch --gpus="0,1,2,3" --log_dir=log_videoswin_base /home/aistudio/data/PaddleVideo-develop/main.py --amp --validate -c /home/aistudio/data/configs/violence_detection/videoswin_violence.yaml

[06/02 21:38:41] epoch:[ 13/30 ] train step:420  loss: 0.00017 lr: 0.000020 top1: 1.00000 top5: 1.00000 batch_cost: 0.14440 sec, reader_cost: 0.00019 sec, ips: 6.92512 instance/sec, eta: 0:02:18, max_mem_reserved: 14413.22 MB max_mem_allocated: 11900.33 MB
[06/02 21:38:45] epoch:[ 13/30 ] train step:440  loss: 0.02944 lr: 0.000020 top1: 1.00000 top5: 1.00000 batch_cost: 0.14815 sec, reader_cost: 0.00019 sec, ips: 6.75009 instance/sec, eta: 0:02:22, max_mem_reserved: 14413.22 MB max_mem_allocated: 11900.33 MB
[06/02 21:38:49] epoch:[ 13/30 ] train step:460  loss: 0.28583 lr: 0.000020 top1: 1.00000 top5: 1.00000 batch_cost: 0.19890 sec, reader_cost: 0.00018 sec, ips: 5.02766 instance/sec, eta: 0:02:27, max_mem_reserved: 14413.22 MB max_mem_allocated: 11900.33 MB
[06/02 21:38:52] epoch:[ 13/30 ] train step:480  loss: 0.00005 lr: 0.000020 top1: 1.00000 top5: 1.00000 batch_cost: 0.19580 sec, reader_cost: 0.00015 sec, ips: 5.10725 instance/sec, eta: 0:02:32, max_mem_reserved: 14413.22 MB max

## test
>由于Video-Swin-Transformer模型测试模式的采样方式是速度稍慢但精度高一些的UniformCrop，与训练过程中验证模式采用的CenterCrop不同，所以训练日志中记录的验证指标topk Acc不代表最终的测试分数，因此在训练完成之后可以用测试模式对指定的模型进行测试获取最终的指标

In [6]:
! export CUDA_VISIBLE_DEVICES=0,1,2,3
! python3.10 -B -m paddle.distributed.launch --gpus="0,1,2,3" --log_dir=log_videoswin_base_test  /home/aistudio/data/PaddleVideo-develop/main.py  -c /home/aistudio/data/configs/violence_detection/videoswin_violence_test.yaml --test --weight=/home/aistudio/huitong-zhidun/Swin_video/output/VideoSwin_base/VideoSwin_base_best.pdparams

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2025-06-03 00:09:30,136 -----------  Configuration  ----------------------
LAUNCH INFO 2025-06-03 00:09:30,137 auto_cluster_config: 0
LAUNCH INFO 2025-06-03 00:09:30,137 auto_parallel_config: None
LAUNCH INFO 2025-06-03 00:09:30,137 auto_tuner_json: None
LAUNCH INFO 2025-06-03 00:09:30,137 devices: 0,1,2,3
LAUNCH INFO 2025-06-03 00:09:30,137 elastic_level: -1
LAUNCH INFO 2025-06-03 00:09:30,137 elastic_timeout: 30
LAUNCH INFO 2025-06-03 00:09:30,137 enable_gpu_log: True
LAUNCH INFO 2025-06-03 00:09:30,137 gloo_port: 6767
LAUNCH INFO 2025-06-03 00:09:30,137 host: None
LAUNCH INFO 2025-06-03 00:09:30,137 ips: None
LAUNCH INFO 2025-